# JEV labelling (pre-step for Phase 2)

Jev (TypeSafe) is a System One decision model: it takes a state plus typed questions and returns
typed answers with probabilities. No text is generated. It is served through OpenRouter's
Decisions API (`POST https://openrouter.ai/api/alpha/decisions`), model `typesafe/jev-1.14`.

Here it is the third labeller, beside the two human coders, on the same taxonomy
(`taxonomy.json`) and coding guide (`coding_guide.md`). One call per episode asks six questions:

| question | type | returns |
|---|---|---|
| `is_contestation` | noul | P(yes) |
| `presence`, `target`, `interaction_act`, `grounds`, `expected_response` | choice | chosen value, confidence, probability per value |

All six are answered in parallel and cannot see each other, so the five structure questions are
asked unconditionally and blanked when `is_contestation = no`. The probabilities are kept in the
cache: P(yes) is the detector confidence that E4 needs.

1. Score Jev against the 76 adjudicated gold episodes.
2. Label the 250 episodes in `data/episodes_to_label.csv`.
3. Optionally write the labels into `labels.db` as annotator `jev`.

Labels are cached to `jev/jev_labels.csv` after every episode, so an interrupted run resumes.

**Key.** Put `OPENROUTER_API_KEY=...` in the environment or in `.env` at the project root
(gitignored). Jev is billed on input tokens only; the whole run is a few cents.

In [1]:
import json
import os
import time
from pathlib import Path

import pandas as pd
import requests
from sklearn.metrics import cohen_kappa_score

ROOT = Path.cwd().parent if Path.cwd().name == "jev" else Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "jev"
CACHE = OUT / "jev_labels.csv"

JEV_URL = "https://openrouter.ai/api/alpha/decisions"
JEV_MODEL = "typesafe/jev-1.14"   # pinned; ~typesafe/jev-latest tracks the newest release
YES_THRESHOLD = 0.5               # P(contestation) at or above this → yes
HESITATION = (0.35, 0.65)         # P(contestation) in this band → notes record the probabilities

TAXONOMY = json.loads((ROOT / "taxonomy.json").read_text())
CODING_GUIDE = (ROOT / "coding_guide.md").read_text()

DIMS = {d["name"]: d["values"] + ([d["value_if_no"]] if "value_if_no" in d else [])
        for d in TAXONOMY["dimensions"] if isinstance(d["values"], list)}
STRUCT = ["target", "interaction_act", "grounds", "expected_response"]
CHOICE_DIMS = ["presence", *STRUCT]
LABEL_COLS = ["is_contestation", "presence", *STRUCT, "notes"]
CACHE_COLS = ["episode_id", "split", *LABEL_COLS, "p_contestation", "confidence_labels",
              "answers", "model", "cost", "errors"]

gold = pd.read_csv(DATA / "gold_standard.csv", keep_default_na=False)
todo = pd.read_csv(DATA / "episodes_to_label.csv", keep_default_na=False)
print(f"gold: {len(gold)}  to label: {len(todo)}")
gold[["participant_group", "explanation_format", "is_contestation"]].value_counts().unstack(fill_value=0)

gold: 76  to label: 250


is_contestation                           no  yes
participant_group explanation_format             
SSH               LIME                     5    1
                  SHAP                     1    5
                  Counterfactual           3    3
                  Anchor                   3    3
                  Descriptive statistics   1    4
DE                LIME                     0    3
                  SHAP                     2    4
                  Counterfactual           0    4
                  Anchor                   2    2
                  Descriptive statistics   0    4
IT                LIME                     4    2
                  SHAP                     4    2
                  Counterfactual           1    3
                  Anchor                   5    1
                  Descriptive statistics   1    3

In [2]:
def load_key() -> str:
    key = os.environ.get("OPENROUTER_API_KEY", "")
    env_file = ROOT / ".env"
    if not key and env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith("OPENROUTER_API_KEY="):
                key = line.split("=", 1)[1].strip().strip("\"'")
    if not key:
        raise RuntimeError("set OPENROUTER_API_KEY in the environment or in .env at the project root")
    return key


API_KEY = load_key()
print("key loaded:", API_KEY[:8] + "…")

key loaded: sk-or-v1…


## Questions

The option descriptions are the ones the human coders see in the app's Guide tab
(`app/views/guide.py`). Option keys are snake_case for the API and mapped back to the taxonomy's
values.

In [3]:
DESCRIPTIONS = {
    "presence": {
        "explicit": "The challenge is stated directly and unambiguously. 'This makes no sense.'",
        "implicit": "The challenge is conveyed through tone, phrasing, or implication without being stated outright.",
        "ambiguous": "You cannot tell whether it is a genuine challenge or mere incomprehension / uncertainty.",
    },
    "target": {
        "prediction": "The model's output / classification for this instance.",
        "reasoning": "How the model arrived at its output.",
        "evidence": "The features or data the explanation uses to justify the output.",
        "input data": "The quality, completeness, or correctness of the underlying data.",
        "explanation representation": "The chart or visual itself, when the claim is that it prevents correct interpretation.",
        "system competence": "The AI's general ability or authority to judge in this domain.",
    },
    "interaction_act": {
        "request for justification": "Asks why: 'Why does this feature matter here?'",
        "direct challenge": "Outright assertion that the output or reasoning is wrong.",
        "correction": "Offers the right answer or fixes an error.",
        "alternative proposal": "Suggests a different model, feature, or approach.",
        "counterexample": "Gives a concrete case that contradicts the system's claim.",
        "rejection": "Dismisses the output or explanation without offering an alternative.",
        "request to acknowledge uncertainty": "Asks the system to be less confident.",
    },
    "grounds": {
        "domain knowledge": "Expertise about mushrooms / biology.",
        "technical knowledge": "Expertise about ML, statistics, or XAI methods.",
        "prior experience": "Personal experience collecting or identifying mushrooms.",
        "internal inconsistency": "The explanation contradicts itself.",
        "missing evidence": "Important features or information are absent from the explanation.",
        "data-quality concern": "The underlying data is wrong, incomplete, or unreliable.",
        "causal implausibility": "The feature-outcome relationship doesn't make biological sense.",
        "presentation ambiguity": "The visual or wording is ambiguous in a way that affects interpretation.",
    },
    "expected_response": {
        "justify": "Explain why the model made this decision.",
        "revise": "Change or update the model / output.",
        "incorporate correction": "Accept and act on the participant's correction.",
        "provide additional information": "Supply missing context or data.",
        "acknowledge uncertainty": "Admit the model is not confident here.",
        "expose conflicting evidence": "Show the evidence that conflicts with the participant's claim.",
        "defer to human expertise": "Yield to the participant's domain authority.",
    },
}
for dim in CHOICE_DIMS:   # every taxonomy value must have a description
    assert set(DESCRIPTIONS[dim]) == {v for v in DIMS[dim] if v != "absent"}, dim

INSTRUCTIONS = {
    "presence": "If the target line is contestation, how explicit is the challenge?",
    "target": "If the target line is contestation, what is being challenged?",
    "interaction_act": "If the target line is contestation, what move is the participant making?",
    "grounds": "If the target line is contestation, what is the challenge based on?",
    "expected_response": "If the target line is contestation, what would resolve the challenge?",
}


def key_of(value: str) -> str:
    return value.replace("-", " ").replace(" ", "_")


VALUE_OF = {dim: {key_of(v): v for v in DESCRIPTIONS[dim]} for dim in CHOICE_DIMS}

QUESTIONS = {
    "is_contestation": {
        "type": "noul",
        "instructions": f"{TAXONOMY['definition']} {TAXONOMY['core_test']} {TAXONOMY['unit_rule']}",
        "criteria": {
            "true": "The participant is disputing what the system asserts or does, in the target line "
                    "or its immediate continuation, and calls for justification, correction, "
                    "reconsideration, or acknowledgment of uncertainty.",
            "false": "No dispute: incomprehension, a stated preference about looks, personal "
                     "uncertainty, idle speculation, general affect, or the interviewer's task question.",
        },
    },
    **{
        dim: {
            "type": "choice",
            "instructions": INSTRUCTIONS[dim],
            "criteria": {key_of(v): desc for v, desc in DESCRIPTIONS[dim].items()},
        }
        for dim in CHOICE_DIMS
    },
}


def build_state(ep: pd.Series) -> dict:
    return {
        "task": "Code one target line from a think-aloud interview about an AI mushroom classifier "
                "and its explanations, following the codebook. Polish is authoritative; English is a "
                "translation. Context lines are tagged INT: (interviewer) and P: (participant).",
        "codebook": {
            "definition": TAXONOMY["definition"],
            "core_test": TAXONOMY["core_test"],
            "unit_rule": TAXONOMY["unit_rule"],
            "boundary_rules": TAXONOMY["boundary_rules"],
            "also_no": TAXONOMY["also_no"],
            "coding_guide": CODING_GUIDE,
        },
        "episode": {
            "episode_id": ep["episode_id"],
            "speaker_of_target_line": ep["speaker"],
            "participant_group": ep["participant_group"],
            "explanation_format": ep["explanation_format"],
            "explanation_theme": ep["explanation_theme"],
            "context_pl": ep["context_pl"],
            "target_line_pl": ep["text_pl"],
            "context_en": ep["context_en"],
            "target_line_en": ep["text_en"],
        },
    }


# Jev switches to a slower, less accurate long-context mode above ~32,000 bytes of text
state_bytes = len(json.dumps(build_state(gold.iloc[0]), ensure_ascii=False).encode())
print(f"state for one episode: {state_bytes:,} bytes")
assert state_bytes < 32_000

state for one episode: 10,870 bytes


## Jev call

In [4]:
def jev_decide(state: dict, questions: dict, tries: int = 5) -> dict:
    body = {"model": JEV_MODEL, "state": state, "questions": questions}
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
               "X-Title": "XAI contestation labelling"}
    for attempt in range(tries):
        try:
            r = requests.post(JEV_URL, headers=headers, json=body, timeout=60)
        except requests.RequestException as e:
            if attempt == tries - 1:
                raise
            time.sleep(2 ** (attempt + 1))
            continue
        if r.status_code == 429 or r.status_code >= 500:
            if attempt == tries - 1:
                r.raise_for_status()
            time.sleep(2 ** (attempt + 1))
            continue
        if r.status_code != 200:
            raise RuntimeError(f"Jev {r.status_code}: {r.text[:500]}")
        return r.json()


def jev_label(episode: pd.Series) -> dict:
    res = jev_decide(build_state(episode), QUESTIONS)
    ans = res["answers"]
    p_yes = float(ans["is_contestation"]["noul"])
    lab = {"is_contestation": "yes" if p_yes >= YES_THRESHOLD else "no", "p_contestation": round(p_yes, 4)}
    confs = []
    for dim in CHOICE_DIMS:
        a = ans[dim]
        lab[dim] = VALUE_OF[dim][a["choice"]]
        confs.append(float(a["confidence"]))
    lab["confidence_labels"] = round(min(confs), 4)
    hesitant = (HESITATION[0] <= p_yes <= HESITATION[1]
                or (lab["is_contestation"] == "yes" and lab["presence"] == "ambiguous"))
    lab["notes"] = (
        f"jev P(contestation)={p_yes:.2f}; presence " +
        ", ".join(f"{VALUE_OF['presence'][k]} {v:.2f}" for k, v in ans["presence"]["probabilities"].items())
    ) if hesitant else ""
    lab["answers"] = json.dumps(ans, ensure_ascii=False)
    lab["model"] = res.get("model", JEV_MODEL)
    lab["cost"] = res.get("usage", {}).get("cost", "")
    return lab

In [5]:
# One live call, to see the raw answer before running everything
_ep = gold[gold["gold_role"] == "example"].iloc[0]
print(_ep["episode_id"], "| gold:", _ep["is_contestation"], _ep["presence"], _ep["target"],
      _ep["interaction_act"], _ep["grounds"], _ep["expected_response"])
_lab = jev_label(_ep)
print({k: v for k, v in _lab.items() if k != "answers"})
json.loads(_lab["answers"])

PK_DE_03:64 | gold: yes explicit prediction direct challenge domain knowledge incorporate correction
{'is_contestation': 'no', 'p_contestation': 0.24, 'presence': 'ambiguous', 'target': 'explanation representation', 'interaction_act': 'counterexample', 'grounds': 'presentation ambiguity', 'expected_response': 'provide additional information', 'confidence_labels': 0.31, 'notes': '', 'model': 'typesafe/jev-1.14-20261014', 'cost': 0.00020034}


{'is_contestation': {'type': 'noul', 'noul': 0.24},
 'presence': {'type': 'choice',
  'choice': 'ambiguous',
  'probabilities': {'explicit': 0.04, 'implicit': 0.12, 'ambiguous': 0.84},
  'confidence': 0.76},
 'target': {'type': 'choice',
  'choice': 'explanation_representation',
  'probabilities': {'prediction': 0.05,
   'reasoning': 0.22,
   'evidence': 0.13,
   'input_data': 0.04,
   'explanation_representation': 0.55,
   'system_competence': 0.01},
  'confidence': 0.46},
 'interaction_act': {'type': 'choice',
  'choice': 'counterexample',
  'probabilities': {'request_for_justification': 0.06,
   'direct_challenge': 0.33,
   'correction': 0.13,
   'alternative_proposal': 0,
   'counterexample': 0.41,
   'rejection': 0.06,
   'request_to_acknowledge_uncertainty': 0.01},
  'confidence': 0.31},
 'grounds': {'type': 'choice',
  'choice': 'presentation_ambiguity',
  'probabilities': {'domain_knowledge': 0.03,
   'technical_knowledge': 0,
   'prior_experience': 0,
   'internal_inconsistenc

## Validation against the taxonomy constraints

In [6]:
def validate(lab: dict) -> list[str]:
    errs = []
    for dim, values in DIMS.items():
        v = lab.get(dim, "")
        if v and v not in values:
            errs.append(f"{dim}={v!r} not in taxonomy")
    if lab.get("is_contestation") not in ("yes", "no"):
        errs.append("is_contestation must be yes/no")
    elif lab["is_contestation"] == "no":
        if lab.get("presence") not in ("", "absent"):
            errs.append("presence must be absent when is_contestation=no")
        if any(lab.get(d) for d in STRUCT):
            errs.append("structured dimensions must be empty when is_contestation=no")
    else:
        if lab.get("presence") in ("", "absent"):
            errs.append("presence must be set when is_contestation=yes")
        errs += [f"{d} missing" for d in STRUCT if not lab.get(d)]
    if lab.get("presence") == "ambiguous" and not lab.get("notes"):
        errs.append("notes required when presence=ambiguous")
    return errs


def normalise(lab: dict) -> dict:
    lab = {k: ("" if v is None else v) for k, v in lab.items()}
    if lab.get("is_contestation") == "no":
        lab["presence"] = "absent"
        lab.update({d: "" for d in STRUCT})
    return lab

## Run with cache

In [7]:
def load_cache() -> pd.DataFrame:
    return pd.read_csv(CACHE, keep_default_na=False) if CACHE.exists() else pd.DataFrame()


def run(episodes: pd.DataFrame, split: str) -> pd.DataFrame:
    cache = load_cache()
    done = set(cache.loc[cache["split"] == split, "episode_id"]) if len(cache) else set()
    for i, (_, ep) in enumerate(episodes.iterrows()):
        if ep["episode_id"] in done:
            continue
        lab = normalise(jev_label(ep))
        row = {"episode_id": ep["episode_id"], "split": split, **lab,
               "errors": "; ".join(validate(lab))}
        pd.DataFrame([row]).reindex(columns=CACHE_COLS, fill_value="").to_csv(
            CACHE, mode="a", header=not CACHE.exists(), index=False)
        print(f"\r{split}: {i + 1}/{len(episodes)}", end="")
    print()
    cache = load_cache()
    res = cache[cache["split"] == split].drop_duplicates("episode_id", keep="last")
    n_bad = (res["errors"] != "").sum()
    cost = pd.to_numeric(res["cost"], errors="coerce").sum()
    print(f"{split}: {len(res)} labelled, {n_bad} violate the taxonomy constraints, cost ${cost:.4f}")
    return res

## 1. Agreement with gold (76 adjudicated episodes)

Presence: accuracy and Cohen's κ on `is_contestation`, at the default threshold and swept over
P(contestation). Structure: agreement on each dimension, over episodes where both gold and Jev
say `yes`.

In [8]:
jev_gold = run(gold, "gold")
m = gold.merge(jev_gold, on="episode_id", suffixes=("_gold", "_jev"))

y_g, y_j = m["is_contestation_gold"], m["is_contestation_jev"]
print(f"is_contestation @ {YES_THRESHOLD}: acc={(y_g == y_j).mean():.3f}  "
      f"kappa={cohen_kappa_score(y_g, y_j):.3f}  n={len(m)}")
print(pd.crosstab(y_g, y_j, rownames=["gold"], colnames=["jev"]))

p = pd.to_numeric(m["p_contestation"])
sweep = pd.DataFrame([{
    "threshold": t,
    "acc": (y_g == (p >= t).map({True: "yes", False: "no"})).mean(),
    "kappa": cohen_kappa_score(y_g, (p >= t).map({True: "yes", False: "no"})),
    "pred_yes": int((p >= t).sum()),
} for t in [0.3, 0.4, 0.5, 0.6, 0.7]]).set_index("threshold").round(3)
print(sweep)

both = m[(y_g == "yes") & (y_j == "yes")]
pd.DataFrame({
    d: {"agreement": (both[f"{d}_gold"] == both[f"{d}_jev"]).mean(), "n": len(both)}
    for d in CHOICE_DIMS
}).T.round(3)

gold: 76/76
gold: 76 labelled, 0 violate the taxonomy constraints, cost $0.0164
is_contestation @ 0.5: acc=0.671  kappa=0.371  n=76
jev   no  yes
gold         
no    28    4
yes   21   23
             acc  kappa  pred_yes
threshold                        
0.3        0.697  0.402        35
0.4        0.684  0.389        30
0.5        0.671  0.371        27
0.6        0.645  0.331        23
0.7        0.618  0.298        17


,agreement,n
presence,0.391,23.0
target,0.304,23.0
interaction_act,0.348,23.0
grounds,0.174,23.0
expected_response,0.087,23.0


In [9]:
# Disagreements on presence, for reading
m.loc[y_g != y_j, ["episode_id", "participant_group", "explanation_format", "text_en",
                   "is_contestation_gold", "p_contestation", "notes"]]

,episode_id,participant_group,explanation_format,text_en,is_contestation_gold,p_contestation,notes
4,PK_DE_03:64,DE,Counterfactual,"Well, I very, very much don't, don't, don't un...",yes,0.26,
5,PK_DE_01:121,DE,LIME,Stipe width at the base... uh... changes color...,yes,0.18,
7,MW_SSH_05:85,SSH,SHAP,"Not at first, because these are such long reco...",yes,0.10,
9,MZ_IT_05:213,IT,SHAP,here exactly such features as stipe height and...,no,0.53,jev P(contestation)=0.53; presence explicit 0....
14,MW_SSH_05:122,SSH,LIME,"Brown stipe color, definitely not, because man...",no,0.81,
16,MK_SSH_02:117,SSH,Descriptive statistics,"So possibly they can't be specific enough, alt...",yes,0.38,jev P(contestation)=0.38; presence explicit 0....
19,PK_DE_09:53,DE,SHAP,But I'd have to adopt a certain assumption tha...,yes,0.07,
32,MZ_SSH_03:543,SSH,Descriptive statistics,"They should kind of be — this one on the left,...",yes,0.13,
35,PK_DE_08:72,DE,Descriptive statistics,Right. This model achieved 99.97% accuracy. Th...,yes,0.05,
36,PK_DE_10:40,DE,SHAP,Stipe height... these are certainly features o...,no,0.63,jev P(contestation)=0.63; presence explicit 0....


## 2. Label the 250 episodes

In [10]:
jev_todo = run(todo, "to_label")
print(jev_todo["is_contestation"].value_counts())
jev_todo.merge(todo[["episode_id", "participant_group", "explanation_format"]], on="episode_id") \
    .pivot_table(index="participant_group", columns="explanation_format",
                 values="is_contestation", aggfunc=lambda s: (s == "yes").sum(), fill_value=0)

to_label: 250/250
to_label: 250 labelled, 0 violate the taxonomy constraints, cost $0.0567
is_contestation
no     161
yes     89
Name: count, dtype: int64


explanation_format,Anchor,Counterfactual,Descriptive statistics,LIME,SHAP
participant_group,,,,,
DE,6,6,18,8,13
IT,1,0,2,1,5
SSH,6,6,5,5,7


## 3. Write to `labels.db` (optional)

Off by default. `labels.db` holds the human annotation work and is not in git, so back it up first.
Labels are append-only: a rerun adds a new version row per episode, as the app does. The
probabilities stay in `jev_labels.csv`; only the six fields and the notes go into the database.

In [11]:
WRITE_DB = False

if WRITE_DB:
    import shutil
    import sqlite3

    db_path = ROOT / "labels.db"
    shutil.copy(db_path, db_path.with_suffix(".db.bak"))
    with sqlite3.connect(db_path) as conn:
        conn.execute("INSERT OR IGNORE INTO annotators (username, password_hash, role) VALUES ('jev', '', 'annotator')")
        ann_id = conn.execute("SELECT id FROM annotators WHERE username='jev'").fetchone()[0]
        for _, r in pd.concat([jev_gold, jev_todo]).iterrows():
            version = conn.execute(
                "SELECT COALESCE(MAX(version), 0) + 1 FROM labels WHERE annotator_id=? AND episode_id=?",
                (ann_id, r["episode_id"]),
            ).fetchone()[0]
            conn.execute(
                "INSERT INTO labels (annotator_id, episode_id, version, is_contestation, presence, target,"
                " interaction_act, grounds, expected_response, notes) VALUES (?,?,?,?,?,?,?,?,?,?)",
                (ann_id, r["episode_id"], version, *[r[c] for c in LABEL_COLS]),
            )
    print("written to labels.db as annotator 'jev'")